# Extraction of Logits for Analysis of HydraBLE Hard

In [1]:
import os
from pathlib import Path


os.chdir(Path.cwd().parents[0])
print("Now in:", Path.cwd())

dataPathProcessed = str(Path.cwd()) + r"\data\csv" + r"\Processed Files\\"
checkPointPath = str(Path.cwd()) + r"\out\modeling\checkpoints\\classification_hard\\"
logitsPath = str(Path.cwd()) + r"\out\modeling\logits_and_labels\\classification_hard\\"

Now in: C:\Users\stsax\OneDrive\Studium\9. Semester\Masterarbeit\Repository


In [2]:
MAX_TOKEN_LENGTH = 1024
MAX_SEQUENCE_LENGTH = 32

In [3]:
import pickle
import pandas as pd
from modeling.BleDataset import BLEStreamDataset
import torch

from bpe.bpe import BleBytePairEncoder

MaskConfigPath = str(Path.cwd()) + r"\data_masking\mask_configs\\"
picklePath = str(Path.cwd()) + r"\out\pickle_objects\bpe" + "\\"

with open(picklePath + 'Fitted_BPE_State_Dict.pickle', 'rb') as f:
    state_dict = pickle.load(f)


pkt_df_test = pd.read_parquet(dataPathProcessed + r"classification_dataset_hard_test.parquet")
seq_table_test = pd.read_parquet(dataPathProcessed + r"classification_sequence_table_hard_test.parquet")
stream_idx_test = pd.read_parquet(dataPathProcessed + r"classification_stream_index_hard_test.parquet")



dataset_test = BLEStreamDataset(packet_df=pkt_df_test, sequence_table=seq_table_test, stream_index=stream_idx_test,
                           max_sequence_length=MAX_SEQUENCE_LENGTH, max_token_length=MAX_TOKEN_LENGTH,
                               tokenizer_dict=state_dict, mask_config_path=MaskConfigPath, dataset_type = 'test')

pkt_df_val = pd.read_parquet(dataPathProcessed + r"classification_dataset_hard_validation.parquet")
seq_table_val = pd.read_parquet(dataPathProcessed + r"classification_sequence_table_hard_validation.parquet")
stream_idx_val = pd.read_parquet(dataPathProcessed + r"classification_stream_index_hard_validation.parquet")

dataset_val = BLEStreamDataset(packet_df=pkt_df_val, sequence_table=seq_table_val, stream_index=stream_idx_val,
                           max_sequence_length=MAX_SEQUENCE_LENGTH, max_token_length=MAX_TOKEN_LENGTH,
                           tokenizer_dict=state_dict, mask_config_path=MaskConfigPath, dataset_type = 'validation')

test_loader = torch.utils.data.DataLoader(dataset_test, batch_size=50, num_workers=8, pin_memory=True,     persistent_workers=True,     drop_last=False,     prefetch_factor=4, shuffle=False)

validation_loader = torch.utils.data.DataLoader(dataset_val, batch_size=50, num_workers=8, pin_memory=True,     persistent_workers=True,     drop_last=False,     prefetch_factor=4, shuffle=False)



In [4]:
import torch

from modeling.HydraBLE import HydraBLETransformer

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
bpe = BleBytePairEncoder.from_state_dict(state_dict)

model = HydraBLETransformer(
        vocab_size=bpe.vocab_size,
        num_classes=dataset_test.num_of_known_classes,
        pad_id=1,
        max_heads=8,
        head_dim=64,
        depth=4,
        max_len=2048,
        subnet_heads=(1, 2, 4, 8),
        separate_cls_heads=True,
    )


ckpt = torch.load(checkPointPath + "last.pt", map_location=device)
model.load_state_dict(ckpt["model_state_dict"])

model = model.to(device)
model.eval()

HydraBLETransformer(
  (token_embed): Embedding(2048, 512)
  (blocks): ModuleList(
    (0-3): 4 x HydraEncoderBlock(
      (norm1): HydraLayerNorm()
      (attn): HydraAttention(
        (qkv): HydraLinear()
        (proj): HydraLinear()
        (attn_drop): Dropout(p=0.0, inplace=False)
        (proj_drop): Dropout(p=0.0, inplace=False)
      )
      (norm2): HydraLayerNorm()
      (mlp): HydraMlp(
        (fc1): HydraLinear()
        (fc2): HydraLinear()
        (drop): Dropout(p=0.0, inplace=False)
      )
    )
  )
  (norm): HydraLayerNorm()
  (lm_head): HydraLinear()
  (cls_heads): ModuleDict(
    (1): HydraLinear()
    (2): HydraLinear()
    (4): HydraLinear()
    (8): HydraLinear()
  )
)

In [5]:
from tqdm import tqdm


model.eval()

for h in (1, 2, 4, 8):
    all_logits_val = []
    all_labels_val = []
    all_label_ids = []
    with torch.no_grad():
        for batch in tqdm(validation_loader):
            tokens = batch["tokens"].to(device)
            label = batch["label"]
            label_id = batch["label_id"].to(device)

            model.set_active_heads(h)
            logits = model(tokens)

            all_logits_val.append(logits.cpu())
            all_label_ids.append(label_id.cpu())
            all_labels_val.extend(label)


    all_logits = torch.cat(all_logits_val, dim=0)
    all_label_ids = torch.cat(all_label_ids, dim=0)
    all_labels = all_labels_val

    torch.save(all_logits, logitsPath + f"logits_classification_h={h}_hard_validation.pt")
    torch.save(all_label_ids, logitsPath + f"label_ids_classification_h={h}_hard_validation.pt")

    with open(picklePath + f"labels_classification_h={h}_hard_validation" + '.pickle', 'wb') as f:
        pickle.dump(all_labels, f)


100%|██████████| 128/128 [02:13<00:00,  1.04s/it]


In [6]:
from tqdm import tqdm


model.eval()
for h in (1, 2, 4, 8):
    all_logits_val = []
    all_labels_val = []
    all_label_ids = []
    with torch.no_grad():
        for batch in tqdm(test_loader):
            tokens = batch["tokens"].to(device)
            label = batch["label"]
            label_id = batch["label_id"].to(device)

            model.set_active_heads(h)
            logits = model(tokens)

            all_logits_val.append(logits.cpu())
            all_label_ids.append(label_id.cpu())
            all_labels_val.extend(label)


    all_logits = torch.cat(all_logits_val, dim=0)
    all_label_ids = torch.cat(all_label_ids, dim=0)
    all_labels = all_labels_val

    torch.save(all_logits, logitsPath + f"logits_classification_h={h}_hard_test.pt")
    torch.save(all_label_ids, logitsPath + f"label_ids_classification_h={h}_hard_test.pt")

    with open(picklePath + f"labels_classification_h={h}_hard_test" + '.pickle', 'wb') as f:
        pickle.dump(all_labels, f)

100%|██████████| 128/128 [02:11<00:00,  1.03s/it]
